# 05. Pipeline RAG Completo (Recuperação + Geração Fundamentada em Evidências)

**Contexto:** Tech Challenge 04 - Voice of Customer Intelligence com RAG  
**Requisitos do Desafio:**
- Demonstração completa do fluxo: Pergunta -> Transformação -> Recuperação Semântica -> Contexto -> **Geração com Modelo de Linguagem** fundamentada em evidências.
- Respostas estritamente ancoradas nas avaliações reais dos clientes (Voice of Customer).
- Prevenção ativa de alucinações e tratamento de perguntas fora do escopo ou sem dados suficientes.

**Arquitetura de Geração Plugável (interface `GeradorResposta`):**
- **`local`** (padrão) — `SintetizadorLocalVoC`: síntese por regras, **100% offline, sem chaves e sem custo**.
- **`gemini`** — `GeradorGemini`: LLM de nuvem (Google Gemini) para geração em linguagem natural, **opt-in** via variável de ambiente `GEMINI_API_KEY`, com **degradação graciosa** para o gerador local em caso de indisponibilidade (sem SDK/chave/rede).

In [ ]:
import sys
import os
sys.path.append('../')

from src.rag_pipeline import PipelineRAG, SintetizadorLocalVoC, GeradorGemini, DEFAULT_PROMPT_TEMPLATE
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv(usecwd=True))

DB_PATH = '../data/vector_db/'

# Seleção do motor de geração ("G" do RAG).
# - 'local'  : 100% offline, por regras (padrão).
# - 'gemini' : LLM de nuvem; requer `pip install google-genai` e a variável de ambiente
#              GEMINI_API_KEY. Sem isso, o pipeline degrada automaticamente para o local.

GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
if not GEMINI_API_KEY:
    raise ValueError('GEMINI_API_KEY não encontrada no .env')

PROVIDER = os.getenv('GEN_PROVIDER')
pipeline_rag = PipelineRAG(db_path=DB_PATH, provider=PROVIDER)

print(f"Pipeline RAG conectado ao ChromaDB local! Motor de geração: {type(pipeline_rag.gerador).__name__}")
print(f"Total de documentos indexados na base: {pipeline_rag.collection.count():,}")

# Diagnóstico do provider de nuvem (quando selecionado)
if isinstance(pipeline_rag.gerador, GeradorGemini):
    if pipeline_rag.gerador.disponivel:
        print(f"Gemini ativo: modelo '{pipeline_rag.gerador.model_name}'.")
    else:
        print(f"Gemini indisponível ({pipeline_rag.gerador.erro_init}). Usando fallback local automaticamente.")

ValueError: GEMINI_API_KEY não encontrada no .env

In [ ]:
PROVIDER = "local"

pipeline_rag = PipelineRAG(
    db_path=DB_PATH,
    provider=PROVIDER
)

print("Pipeline RAG inicializado com gerador local!")

modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

c:\Users\Usuário\Downloads\1AIST_projects-feature-tc04\tech-challenge-04\tech-challenge-fase4\.venv\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Usuário\.cache\huggingface\hub\models--sentence-transformers--paraphrase-multilingual-MiniLM-L12-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (6 outputs left out)

tokenizer.json: downloading bytes:           |  0.00B            

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

ValueError: An embedding function already exists in the collection configuration, and a new one is provided. If this is intentional, please embed documents separately. Embedding function conflict: new: sentence_transformer vs persisted: default

In [2]:
import chromadb
print(chromadb.__version__)

1.5.9


## 1. Engenharia de Prompt e Diretrizes de Fidelidade (Anti-Alucinação)
Visualizamos o template de prompt que orienta a síntese analítica e impõe a recusa caso as evidências não sustentem a resposta.

In [ ]:
print("=== TEMPLATE DE PROMPT E REGRAS DE GROUNDING ===")
print(DEFAULT_PROMPT_TEMPLATE)

=== TEMPLATE DE PROMPT E REGRAS DE GROUNDING ===

Você é um especialista em Inteligência de Voz do Cliente (Voice of Customer Intelligence) analisando avaliações de e-commerce.
Sua missão é responder à pergunta do gestor ESTRITAMENTE com base nas Evidências Recuperadas abaixo.

EVIDÊNCIAS RECUPERADAS DA BASE DE CONHECIMENTO:
{context}

DIRETRIZES OBRIGATÓRIAS:
1. FIDELIDADE AOS DADOS: Responda apenas com fatos sustentados pelos comentários acima. Nunca invente ou assuma informações externas.
2. RASTREABILIDADE E EVIDÊNCIAS: Cite trechos ou menções dos comentários que fundamentam cada insight apresentado.
3. TRATAMENTO DE INSUFICIÊNCIA: Se as evidências recuperadas forem insuficientes ou a pergunta estiver fora do escopo da base de clientes, responda EXATAMENTE:
   "Não possuo evidências suficientes na base de conhecimento para responder a esta pergunta."

Pergunta do Gestor: {question}

Resposta Fundamentada do Analista:



## 2. Cenário 1: Investigação de Reclamações de Entrega (Edital)
Pergunta de negócio: *"O que os clientes relatam sobre problemas relacionados à entrega?"*  
Utilizamos o **Filtro de Metadados** (`review_score <= 2`) para recuperar estritamente experiências insatisfatórias.

In [ ]:
pergunta_1 = "O que os clientes relatam sobre problemas relacionados à entrega?"
resultado_1 = pipeline_rag.responder(
    pergunta=pergunta_1,
    n_results=4,
    where_filter={"review_score": {"$lte": 2}}
)

print(f"PERGUNTA: {resultado_1['pergunta']}\n")
print(resultado_1['resposta'])

PERGUNTA: O que os clientes relatam sobre problemas relacionados à entrega?

Com base estritamente nas evidências recuperadas, os clientes relatam os seguintes pontos sobre problemas relacionados à entrega:

1. **Atrasos e Falta de Suporte/Tratamento:**
   * Há relatos de atraso no cumprimento dos prazos e ausência de suporte para solucionar o problema. Conforme apontado na **Evidência 1**, o cliente reclama de *"falta de consideração com o cliente, pois ocorreu problema na entrega é em momento nenhum o problema foi tratado"* e menciona que fez *"duas compras, uma com prazo de entrega e , as atrasou"*.

2. **Falta de Comunicação sobre Dificuldades:**
   * Os consumidores apontam que a loja falha em comunicar problemas operacionais ou imprevistos na entrega. Na **Evidência 4**, o cliente destaca que *"se houve alguma dificuldade para entrega do produto, no mínimo, o comprador deveria ser avisado e negociado uma nova situação de entrega"*.

3. **Incoerência Logística e Dificuldades Criad

## 3. Cenário 2: Busca por Experiências Positivas e Elogios (Edital)
Pergunta de negócio: *"Quais aspectos da experiência de compra são mais elogiados pelos clientes?"*  
Filtramos avaliações promotoras (`review_score >= 4`).

In [ ]:
pergunta_2 = "Quais aspectos da experiência de compra são mais elogiados pelos clientes?"
resultado_2 = pipeline_rag.responder(
    pergunta=pergunta_2,
    n_results=4,
    where_filter={"review_score": {"$gte": 4}}
)

print(f"PERGUNTA: {resultado_2['pergunta']}\n")
print(resultado_2['resposta'])

PERGUNTA: Quais aspectos da experiência de compra são mais elogiados pelos clientes?

Com base nas avaliações analisadas, os principais aspectos da experiência de compra elogiados pelos clientes são:

1. **Agilidade na Entrega e Condição dos Produtos:**
   * Os clientes destacam a rapidez no envio e a integridade dos itens. Na **Evidência 2**, o consumidor elogia a entrega antecipada e a qualidade: *"o produto desta compra chegou antes do prazo maximo para entrega"* e relata que *"sempre recebi em perfeitas condições"*. O tema também é associado a uma experiência positiva na **Evidência 4**, cujo título é *"entrega"* e o comentário afirma: *"foi poditiva a experiência de compra"*.

2. **Qualidade do Atendimento e Compromisso do Vendedor:**
   * A conduta e o suporte da loja foram positivamente avaliados. Na **Evidência 2**, o vendedor é classificado como *"excelente vendedor"*. Já na **Evidência 3**, sob o título *"bom serviço"*, o cliente agradece nominalmente pelo suporte e postura d

## 4. Cenário 3: Tratamento de Perguntas sem Evidências Suficientes / Fora de Escopo
Conforme o Item 4 do edital: *"Soluções baseadas em IA Generativa podem produzir respostas aparentemente convincentes mesmo quando não existem informações suficientes para sustentá-las. Sua solução deverá considerar situações em que a pergunta esteja fora do escopo da base de conhecimento."*  

Testamos uma pergunta esdrúxula: *"Qual a cor do uniforme do entregador e qual brinde de chocolate foi enviado no pacote?"*.

In [ ]:
pergunta_fora_escopo = "Qual a cor do uniforme do entregador e qual brinde de chocolate foi enviado no pacote?"
resultado_3 = pipeline_rag.responder(
    pergunta=pergunta_fora_escopo,
    n_results=3
)

print(f"PERGUNTA: {resultado_3['pergunta']}\n")
print(f"RESPOSTA DO SISTEMA:")
print(resultado_3['resposta'])

PERGUNTA: Qual a cor do uniforme do entregador e qual brinde de chocolate foi enviado no pacote?

RESPOSTA DO SISTEMA:
Não possuo evidências suficientes na base de conhecimento para responder a esta pergunta.


## 5. Rastreabilidade das Evidências (Auditoria Completa)
O sistema permite auditar detalhadamente todos os metadados, identificadores (`review_id`) e categorias retornadas pela base vetorial.

In [ ]:
print("Auditoria dos IDs recuperados para a Pergunta 1:")
for i, (doc_id, meta) in enumerate(zip(resultado_1['ids'], resultado_1['metadados']), 1):
    print(f"[{i}] ID do Review: {doc_id} | Nota no BD: {meta.get('review_score')}")

Auditoria dos IDs recuperados para a Pergunta 1:
[1] ID do Review: 4baf65ba0f4dc753e9cf3d8863433da0 | Nota no BD: 2
[2] ID do Review: 12416af18a71403380ecd6e6ca53332c | Nota no BD: 1
[3] ID do Review: 66179b1ca078cdb08c6ca238a0a4c654 | Nota no BD: 1
[4] ID do Review: c20dc8f95c7e0a17a4192984287a48ef | Nota no BD: 1


In [ ]:
import pandas
import chromadb
import sentence_transformers
import google.genai
import dotenv

print("Todas as bibliotecas foram carregadas com sucesso!")

Todas as bibliotecas foram carregadas com sucesso!


## 6. Explorações avançadas da Voz do Cliente

Nesta etapa, ampliamos as consultas ao pipeline RAG para explorar padrões de insatisfação, experiências positivas, problemas de entrega e oportunidades de melhoria na experiência dos consumidores.

In [10]:
import os
import sys

sys.path.append(os.path.abspath("../"))

print("Caminho adicionado:", os.path.abspath("../"))
print("Pasta src encontrada:", os.path.exists("../src"))

Caminho adicionado: c:\Users\Usuário\Downloads\1AIST_projects-feature-tc04\tech-challenge-04\tech-challenge-fase4
Pasta src encontrada: True


In [ ]:
from src.rag_pipeline import PipelineRAG

pipeline_rag = PipelineRAG(
    db_path="../data/vector_db/",
    provider="local"
)

print("Pipeline RAG inicializado com sucesso!")

In [12]:
print("client:", "client" in globals())
print("collection:", "collection" in globals())
print("pipeline_rag:", "pipeline_rag" in globals())

client: False
collection: False
pipeline_rag: False


In [9]:
for i, pergunta in enumerate(perguntas_adicionais, 1):
    print("\n" + "=" * 80)
    print(f"PERGUNTA {i}: {pergunta}")
    print("=" * 80)

    resultado = pipeline_rag.responder(pergunta)

    print(resultado)


PERGUNTA 1: Quais são os principais problemas relatados pelos clientes nas avaliações?


NameError: name 'pipeline_rag' is not defined

## 6. Explorações avançadas da Voz do Cliente

Nesta etapa, ampliamos as possibilidades de análise das avaliações por meio de consultas direcionadas ao pipeline RAG. O objetivo é explorar padrões de insatisfação, experiências positivas, problemas recorrentes e oportunidades de melhoria, utilizando os metadados disponíveis para refinar a recuperação das evidências.

In [14]:
exploracoes_rag = [
    {
        "tema": "Principais problemas",
        "pergunta": "Quais são os principais problemas relatados pelos clientes?",
        "filtro": {"nota_min": 1, "nota_max": 2}
    },
    {
        "tema": "Padrões de insatisfação",
        "pergunta": "Quais situações aparecem repetidamente nos relatos de clientes insatisfeitos?",
        "filtro": {"nota_min": 1, "nota_max": 2}
    },
    {
        "tema": "Experiência de entrega",
        "pergunta": "Quais problemas relacionados à entrega são mencionados pelos consumidores?",
        "filtro": {}
    },
    {
        "tema": "Elogios dos consumidores",
        "pergunta": "Quais aspectos da experiência de compra são mais elogiados?",
        "filtro": {"nota_min": 4, "nota_max": 5}
    },
    {
        "tema": "Análise por categoria",
        "pergunta": "Quais aspectos positivos e negativos são destacados pelos clientes nesta categoria?",
        "filtro": {"categoria": "beleza_saude"}
    },
    {
        "tema": "Comparação de avaliações",
        "pergunta": "Quais diferenças aparecem entre as experiências relatadas em avaliações positivas e negativas?",
        "filtro": {}
    },
    {
        "tema": "Análise temporal",
        "pergunta": "Quais problemas e elogios aparecem nas avaliações deste período?",
        "filtro": {"ano": 2018}
    },
    {
        "tema": "Oportunidades de melhoria",
        "pergunta": "Com base nos relatos negativos, quais melhorias deveriam ser priorizadas para melhorar a experiência do cliente?",
        "filtro": {"nota_min": 1, "nota_max": 2}
    }
]

In [15]:
for item in exploracoes_rag:
    print(f"{item['tema']}: {item['pergunta']}")
    print(f"Filtros: {item['filtro']}")
    print("-" * 60)

Principais problemas: Quais são os principais problemas relatados pelos clientes?
Filtros: {'nota_min': 1, 'nota_max': 2}
------------------------------------------------------------
Padrões de insatisfação: Quais situações aparecem repetidamente nos relatos de clientes insatisfeitos?
Filtros: {'nota_min': 1, 'nota_max': 2}
------------------------------------------------------------
Experiência de entrega: Quais problemas relacionados à entrega são mencionados pelos consumidores?
Filtros: {}
------------------------------------------------------------
Elogios dos consumidores: Quais aspectos da experiência de compra são mais elogiados?
Filtros: {'nota_min': 4, 'nota_max': 5}
------------------------------------------------------------
Análise por categoria: Quais aspectos positivos e negativos são destacados pelos clientes nesta categoria?
Filtros: {'categoria': 'beleza_saude'}
------------------------------------------------------------
Comparação de avaliações: Quais diferenças apar

In [16]:
comparacao_rag = {
    "avaliacoes_negativas": {
        "pergunta": "Quais são os principais problemas e insatisfações relatados pelos consumidores?",
        "filtros": {"nota_min": 1, "nota_max": 2}
    },
    "avaliacoes_positivas": {
        "pergunta": "Quais aspectos da experiência de compra são mais elogiados pelos consumidores?",
        "filtros": {"nota_min": 4, "nota_max": 5}
    }
}

print("Consultas preparadas para comparação:")
for grupo, dados in comparacao_rag.items():
    print(f"\n{grupo.upper()}")
    print("Pergunta:", dados["pergunta"])
    print("Filtros:", dados["filtros"])

Consultas preparadas para comparação:

AVALIACOES_NEGATIVAS
Pergunta: Quais são os principais problemas e insatisfações relatados pelos consumidores?
Filtros: {'nota_min': 1, 'nota_max': 2}

AVALIACOES_POSITIVAS
Pergunta: Quais aspectos da experiência de compra são mais elogiados pelos consumidores?
Filtros: {'nota_min': 4, 'nota_max': 5}


In [17]:
resultados_exploracao = []

for item in exploracoes_rag:
    resultados_exploracao.append({
        "tema": item["tema"],
        "pergunta": item["pergunta"],
        "filtros_aplicados": item["filtro"],
        "resposta": None,
        "evidencias": None
    })

print(f"Total de consultas preparadas: {len(resultados_exploracao)}")
print("Estrutura pronta para receber respostas e evidências do RAG.")

Total de consultas preparadas: 8
Estrutura pronta para receber respostas e evidências do RAG.


## 7. Avaliação qualitativa das respostas do RAG

Para avaliar a qualidade das respostas produzidas pelo pipeline, foi estruturada uma matriz de avaliação considerando quatro critérios: aderência à pergunta, fundamentação nas evidências recuperadas, coerência com os filtros aplicados e ausência de conclusões não sustentadas pelos dados.

In [18]:
criterios_avaliacao = [
    "Aderência à pergunta",
    "Fundamentação nas evidências",
    "Coerência com os filtros",
    "Ausência de conclusões sem evidências"
]

matriz_avaliacao = []

for item in resultados_exploracao:
    matriz_avaliacao.append({
        "tema": item["tema"],
        "aderencia": None,
        "evidencias": None,
        "filtros": None,
        "confiabilidade": None,
        "observacoes": ""
    })

print("Matriz de avaliação criada!")
print("Consultas:", len(matriz_avaliacao))
print("Critérios:", criterios_avaliacao)

Matriz de avaliação criada!
Consultas: 8
Critérios: ['Aderência à pergunta', 'Fundamentação nas evidências', 'Coerência com os filtros', 'Ausência de conclusões sem evidências']


In [19]:
import pandas as pd

df_avaliacao = pd.DataFrame([
    {
        "Tema": item["tema"],
        "Aderência à pergunta": "",
        "Fundamentação nas evidências": "",
        "Coerência com os filtros": "",
        "Ausência de conclusões sem evidências": "",
        "Observações": ""
    }
    for item in resultados_exploracao
])

display(df_avaliacao)

,Tema,Aderência à pergunta,Fundamentação nas evidências,Coerência com os filtros,Ausência de conclusões sem evidências,Observações
0,Principais problemas,,,,,
1,Padrões de insatisfação,,,,,
2,Experiência de entrega,,,,,
3,Elogios dos consumidores,,,,,
4,Análise por categoria,,,,,
5,Comparação de avaliações,,,,,
6,Análise temporal,,,,,
7,Oportunidades de melhoria,,,,,


In [20]:
def calcular_resultado_avaliacao(df):
    criterios = [
        "Aderência à pergunta",
        "Fundamentação nas evidências",
        "Coerência com os filtros",
        "Ausência de conclusões sem evidências"
    ]

    notas = df[criterios].apply(pd.to_numeric, errors="coerce")

    df["Média geral"] = notas.mean(axis=1).round(2)

    return df

print("Função de avaliação criada.")

Função de avaliação criada.


In [21]:
def classificar_qualidade(media):
    if pd.isna(media):
        return "Não avaliada"
    elif media >= 4.5:
        return "Excelente"
    elif media >= 3.5:
        return "Boa"
    elif media >= 2.5:
        return "Regular"
    else:
        return "Precisa melhorar"

print("Classificação automática criada!")

Classificação automática criada!
